# 04 - Ratings per film and per year for the 142 nominees

Inputs: `data/raw/ml-32m/ratings.csv` and `data/processed/best_picture_nominees_1996_2017.csv` (built by `src/build_nominees_table.py`, see notebook 03).

Status: work in progress. The analysis window and the minimum number of ratings are being discussed again.

Window: from the ceremony year to the ceremony year + 5. All years are computed in UTC.

In [1]:
import duckdb

con = duckdb.connect()
con.sql("SET enable_progress_bar = false")  # no progress bar in the saved outputs

## A. A worked example: Titanic (movieId 1721) and platform activity

In [2]:
# Step 7: Titanic (movieId 1721), number of ratings and average rating per year (UTC)
con.sql("SELECT year(to_timestamp(timestamp) AT TIME ZONE 'UTC') AS rating_year, COUNT(*) AS nb_ratings, ROUND(AVG(rating), 2) AS avg_rating FROM '../data/raw/ml-32m/ratings.csv' WHERE movieId = 1721 GROUP BY rating_year ORDER BY rating_year").show(max_rows=40)

┌─────────────┬────────────┬────────────┐
│ rating_year │ nb_ratings │ avg_rating │
│    int64    │   int64    │   double   │
├─────────────┼────────────┼────────────┤
│        1997 │         39 │       4.31 │
│        1998 │       1542 │       4.11 │
│        1999 │       3525 │       3.32 │
│        2000 │       2929 │       3.58 │
│        2001 │       1826 │       3.51 │
│        2002 │       1378 │       3.28 │
│        2003 │       1189 │       3.27 │
│        2004 │       1986 │       3.15 │
│        2005 │       2805 │       3.06 │
│        2006 │       2211 │        3.2 │
│        2007 │       2316 │       3.13 │
│        2008 │       2556 │       3.22 │
│        2009 │       1669 │       3.28 │
│        2010 │       1629 │       3.31 │
│        2011 │       1287 │       3.41 │
│        2012 │       1006 │       3.49 │
│        2013 │        665 │       3.51 │
│        2014 │        528 │        3.4 │
│        2015 │       1878 │       3.44 │
│        2016 │       2881 │      

In [3]:
# Step 8: date and time (UTC) of the very first rating of Titanic (movieId 1721)
con.sql("SELECT to_timestamp(MIN(timestamp)) AT TIME ZONE 'UTC' AS first_rating_utc FROM '../data/raw/ml-32m/ratings.csv' WHERE movieId = 1721")

┌─────────────────────┐
│  first_rating_utc   │
│      timestamp      │
├─────────────────────┤
│ 1997-12-19 19:57:55 │
└─────────────────────┘

In [4]:
# Step 9: total number of ratings per year in the whole dataset (all movies), UTC
con.sql("SELECT year(to_timestamp(timestamp) AT TIME ZONE 'UTC') AS rating_year, COUNT(*) AS nb_ratings FROM '../data/raw/ml-32m/ratings.csv' GROUP BY rating_year ORDER BY rating_year").show(max_rows=40)

┌─────────────┬────────────┐
│ rating_year │ nb_ratings │
│    int64    │   int64    │
├─────────────┼────────────┤
│        1995 │          4 │
│        1996 │    1571368 │
│        1997 │     685388 │
│        1998 │     301691 │
│        1999 │    1174629 │
│        2000 │    1912322 │
│        2001 │    1160098 │
│        2002 │     849762 │
│        2003 │    1011000 │
│        2004 │    1139068 │
│        2005 │    1752573 │
│        2006 │    1141704 │
│        2007 │    1023813 │
│        2008 │    1117071 │
│        2009 │     890696 │
│        2010 │     860073 │
│        2011 │     729301 │
│        2012 │     695493 │
│        2013 │     563301 │
│        2014 │     518537 │
│        2015 │    1743866 │
│        2016 │    1918739 │
│        2017 │    1827953 │
│        2018 │    1391057 │
│        2019 │    1385467 │
│        2020 │    1688159 │
│        2021 │    1239912 │
│        2022 │     913105 │
│        2023 │     794054 │
├─────────────┴────────────┤
│ 29 rows     

## B. The table film x year for all nominees

In [5]:
# Step 11: for each of the 142 nominees and each rating year (UTC): number of ratings and average rating
con.sql("SELECT n.ceremony_year, n.title, n.is_winner, n.movieId, year(to_timestamp(r.timestamp) AT TIME ZONE 'UTC') AS rating_year, COUNT(*) AS nb_ratings, ROUND(AVG(r.rating), 2) AS avg_rating FROM '../data/processed/best_picture_nominees_1996_2017.csv' n JOIN '../data/raw/ml-32m/ratings.csv' r ON n.movieId = r.movieId GROUP BY n.ceremony_year, n.title, n.is_winner, n.movieId, rating_year ORDER BY n.ceremony_year, n.title, rating_year").show(max_rows=30)


┌───────────────┬───────────────────────┬───────────┬─────────┬─────────────┬────────────┬────────────┐
│ ceremony_year │         title         │ is_winner │ movieId │ rating_year │ nb_ratings │ avg_rating │
│     int64     │        varchar        │  boolean  │  int64  │    int64    │   int64    │   double   │
├───────────────┼───────────────────────┼───────────┼─────────┼─────────────┼────────────┼────────────┤
│          1996 │ Apollo 13             │ false     │     150 │        1996 │      19102 │        4.0 │
│          1996 │ Apollo 13             │ false     │     150 │        1997 │       1624 │       3.98 │
│          1996 │ Apollo 13             │ false     │     150 │        1998 │        990 │       4.04 │
│          1996 │ Apollo 13             │ false     │     150 │        1999 │       2024 │        4.0 │
│          1996 │ Apollo 13             │ false     │     150 │        2000 │       2397 │       4.02 │
│          1996 │ Apollo 13             │ false     │     150 │ 

In [6]:
# Step 12: create a reusable view with the number of ratings and average rating per nominee and per rating year (UTC)
con.sql("CREATE VIEW film_year AS SELECT n.ceremony_year, n.title, n.is_winner, n.movieId, year(to_timestamp(r.timestamp) AT TIME ZONE 'UTC') AS rating_year, COUNT(*) AS nb_ratings, ROUND(AVG(r.rating), 2) AS avg_rating FROM '../data/processed/best_picture_nominees_1996_2017.csv' n JOIN '../data/raw/ml-32m/ratings.csv' r ON n.movieId = r.movieId GROUP BY n.ceremony_year, n.title, n.is_winner, n.movieId, rating_year")

## C. Apply the window

The window runs from the ceremony year to the ceremony year + 5 (6 calendar years), the same for every film.
We apply it first; the minimum number of ratings will be discussed afterwards, on the rows that are inside the window.

In [7]:
# Step 1: keep only the rows inside the window, and add the number of years since the ceremony (0 to 5)
con.sql("CREATE VIEW film_year_window AS SELECT *, rating_year - ceremony_year AS years_since_ceremony FROM film_year WHERE rating_year BETWEEN ceremony_year AND ceremony_year + 5")

In [8]:
# Step 2: check the window: how many rows for each year since the ceremony (we expect 142 films each time)
con.sql("SELECT years_since_ceremony, COUNT(*) AS n_rows FROM film_year_window GROUP BY years_since_ceremony ORDER BY years_since_ceremony")

┌──────────────────────┬────────┐
│ years_since_ceremony │ n_rows │
│        int64         │ int64  │
├──────────────────────┼────────┤
│                    0 │    142 │
│                    1 │    142 │
│                    2 │    142 │
│                    3 │    142 │
│                    4 │    142 │
│                    5 │    142 │
└──────────────────────┴────────┘

In [9]:
# Step 3: total number of rows in the window, and number of different films
con.sql("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT movieId) AS n_films FROM film_year_window")

┌────────┬─────────┐
│ n_rows │ n_films │
│ int64  │  int64  │
├────────┼─────────┤
│    852 │     142 │
└────────┴─────────┘

## D. How reliable are the yearly averages? Testing a minimum number of ratings

A yearly average built on few ratings is unreliable. To choose a minimum number of ratings we look at four things, in this order, inside the window only:
1. how the number of ratings per row is distributed,
2. how many rows would fall below several candidate minimums,
3. which films and years those rows are,
4. what precision (error of an average) each minimum gives.

Nothing is removed in this section.

In [10]:
# Step 4: distribution of the number of ratings per row (smallest, quartiles, median, largest)
con.sql("SELECT MIN(nb_ratings) AS smallest, QUANTILE_CONT(nb_ratings, 0.25) AS q25, MEDIAN(nb_ratings) AS median, QUANTILE_CONT(nb_ratings, 0.75) AS q75, MAX(nb_ratings) AS largest FROM film_year_window")

┌──────────┬────────┬────────┬─────────┬─────────┐
│ smallest │  q25   │ median │   q75   │ largest │
│  int64   │ double │ double │ double  │  int64  │
├──────────┼────────┼────────┼─────────┼─────────┤
│       29 │  420.0 │  931.5 │ 1689.25 │   19102 │
└──────────┴────────┴────────┴─────────┴─────────┘

In [11]:
# Step 5: number of rows below several candidate minimums (nothing is removed)
con.sql("SELECT COUNT(*) AS total_rows, SUM(CASE WHEN nb_ratings < 10 THEN 1 ELSE 0 END) AS below_10, SUM(CASE WHEN nb_ratings < 30 THEN 1 ELSE 0 END) AS below_30, SUM(CASE WHEN nb_ratings < 50 THEN 1 ELSE 0 END) AS below_50, SUM(CASE WHEN nb_ratings < 100 THEN 1 ELSE 0 END) AS below_100 FROM film_year_window")

┌────────────┬──────────┬──────────┬──────────┬───────────┐
│ total_rows │ below_10 │ below_30 │ below_50 │ below_100 │
│   int64    │  int128  │  int128  │  int128  │  int128   │
├────────────┼──────────┼──────────┼──────────┼───────────┤
│        852 │        0 │        2 │        6 │        28 │
└────────────┴──────────┴──────────┴──────────┴───────────┘

In [12]:
# Step 6: the rows with fewer than 100 ratings, smallest first (which film, which year)
con.sql("SELECT ceremony_year, title, rating_year, years_since_ceremony, nb_ratings FROM film_year_window WHERE nb_ratings < 100 ORDER BY nb_ratings, ceremony_year, title").show(max_rows=40)

┌───────────────┬─────────────────────────────────────────────────┬─────────────┬──────────────────────┬────────────┐
│ ceremony_year │                      title                      │ rating_year │ years_since_ceremony │ nb_ratings │
│     int64     │                     varchar                     │    int64    │        int64         │   int64    │
├───────────────┼─────────────────────────────────────────────────┼─────────────┼──────────────────────┼────────────┤
│          2012 │ Extremely Loud & Incredibly Close               │        2014 │                    2 │         29 │
│          2012 │ War Horse                                       │        2014 │                    2 │         29 │
│          2011 │ The Kids Are All Right                          │        2014 │                    3 │         38 │
│          2017 │ Fences                                          │        2022 │                    5 │         40 │
│          2010 │ Precious: Based on the Novel 'Push' by

In [13]:
# Step 7: standard deviation of all the ratings given to the 142 nominees (used to compute the error of an average)
con.sql("SELECT ROUND(STDDEV(r.rating), 2) AS std_rating FROM '../data/processed/best_picture_nominees_1996_2017.csv' n JOIN '../data/raw/ml-32m/ratings.csv' r ON n.movieId = r.movieId")

┌────────────┐
│ std_rating │
│   double   │
├────────────┤
│       0.94 │
└────────────┘

In [14]:
# Step 8: the full 6-year block of each of the 13 films that have at least one row below 100 ratings
con.sql("SELECT ceremony_year, title, rating_year, years_since_ceremony, nb_ratings, avg_rating, CASE WHEN nb_ratings < 100 THEN 'below 100' ELSE '' END AS flag FROM film_year_window WHERE movieId IN (SELECT DISTINCT movieId FROM film_year_window WHERE nb_ratings < 100) ORDER BY ceremony_year, title, years_since_ceremony").show(max_rows=100)

┌───────────────┬─────────────────────────────────────────────────┬─────────────┬──────────────────────┬────────────┬────────────┬───────────┐
│ ceremony_year │                      title                      │ rating_year │ years_since_ceremony │ nb_ratings │ avg_rating │   flag    │
│     int64     │                     varchar                     │    int64    │        int64         │   int64    │   double   │  varchar  │
├───────────────┼─────────────────────────────────────────────────┼─────────────┼──────────────────────┼────────────┼────────────┼───────────┤
│          2002 │ In the Bedroom                                  │        2002 │                    0 │        830 │       3.85 │           │
│          2002 │ In the Bedroom                                  │        2003 │                    1 │        537 │       3.64 │           │
│          2002 │ In the Bedroom                                  │        2004 │                    2 │        273 │       3.61 │           │

## E. The weighted slope of one film (Titanic)

The trajectory of a film is summarised by the slope of the line that follows its yearly average ratings: how much the average changes per year. Each year is weighted by its number of ratings. We check the numbers on one film before doing it for the 142 films.

In [15]:
# Step 9: the six yearly rows of Titanic, as a pandas table
titanic = con.sql("SELECT years_since_ceremony, nb_ratings, avg_rating FROM film_year_window WHERE title = 'Titanic' ORDER BY years_since_ceremony").df()
titanic

,years_since_ceremony,nb_ratings,avg_rating
0,0,1542,4.11
1,1,3525,3.32
2,2,2929,3.58
3,3,1826,3.51
4,4,1378,3.28
5,5,1189,3.27


In [16]:
# Step 10: weighted regression: yearly average explained by the years since the ceremony, weights = number of ratings
import statsmodels.api as sm

X = sm.add_constant(titanic["years_since_ceremony"])
result = sm.WLS(titanic["avg_rating"], X, weights=titanic["nb_ratings"]).fit()

print(result.params)
print(result.bse)

const                   3.703187
years_since_ceremony   -0.096330
dtype: float64
const                   0.188466
years_since_ceremony    0.072618
dtype: float64


## F. The weighted slope of each of the 142 films

Same regression as for Titanic, repeated for every film. The result is one slope and its standard error per film.

In [17]:
# Step 11: one weighted regression per film: slope and standard error of the slope
import pandas as pd

rows_all = con.sql("SELECT movieId, years_since_ceremony, nb_ratings, avg_rating FROM film_year_window").df()

results = []
for movie_id, rows in rows_all.groupby("movieId"):
    X = sm.add_constant(rows["years_since_ceremony"])
    fit = sm.WLS(rows["avg_rating"], X, weights=rows["nb_ratings"]).fit()
    results.append({"movieId": movie_id,
                    "slope": fit.params["years_since_ceremony"],
                    "slope_se": fit.bse["years_since_ceremony"]})

slopes = pd.DataFrame(results)
slopes.shape

(142, 3)

In [18]:
# Step 12: add the title, the ceremony year and the winner flag to each slope
films = con.sql("SELECT DISTINCT movieId, title, ceremony_year, is_winner FROM film_year_window").df()
slopes = films.merge(slopes, on="movieId").sort_values(["ceremony_year", "title"]).reset_index(drop=True)
slopes

,movieId,title,ceremony_year,is_winner,slope,slope_se
0,150,Apollo 13,1996,False,-0.003603,0.006176
1,34,Babe,1996,False,-0.011376,0.017138
2,110,Braveheart,1996,True,-0.023663,0.019767
3,17,Sense and Sensibility,1996,False,-0.013033,0.033427
4,58,The Postman (Il Postino),1996,False,-0.014679,0.017118
...,...,...,...,...,...,...
137,166643,Hidden Figures,2017,False,-0.012944,0.013407
138,164909,La La Land,2017,False,-0.002870,0.006662
139,165551,Lion,2017,False,-0.005034,0.013143
140,165549,Manchester by the Sea,2017,False,0.017509,0.006099


In [19]:
# Step 13: checks: 142 films, no missing slope, and Titanic gives the same values as before
print("films:", len(slopes), "| missing slopes:", slopes["slope"].isna().sum())
slopes[slopes["title"] == "Titanic"]

films: 142 | missing slopes: 0


,movieId,title,ceremony_year,is_winner,slope,slope_se
14,1721,Titanic,1998,True,-0.09633,0.072618


In [20]:
# Step 14: summary of the 142 slopes (count, mean, spread, quartiles) and number of positive slopes
print(slopes["slope"].describe().round(4))
print("Positive slopes:", (slopes["slope"] > 0).sum(), "of", len(slopes))

count    142.0000
mean      -0.0265
std        0.0328
min       -0.1170
25%       -0.0492
50%       -0.0258
75%       -0.0020
max        0.0374
Name: slope, dtype: float64
Positive slopes: 32 of 142


## G. Secondary analysis: winners against all the other nominees, ignoring the ceremonies

First look, kept for the record. It puts the 22 winners and the 120 other nominees in two bags and ignores the ceremonies, so it does not remove the differences between eras. The main analysis is section H (comparison inside each ceremony).

In [21]:
# Step 15: number of films, mean slope and spread of the slopes, for the winners (True) and the other nominees (False)
slopes.groupby("is_winner")["slope"].agg(["count", "mean", "std"]).round(4)

,count,mean,std
is_winner,,,
False,120,-0.0238,0.031
True,22,-0.0413,0.039


In [22]:
# Step 16: is the difference between winners and the other nominees larger than chance? (Welch t-test and 95% confidence interval)
from scipy import stats

winners = slopes.loc[slopes["is_winner"], "slope"]
others = slopes.loc[~slopes["is_winner"], "slope"]

test = stats.ttest_ind(winners, others, equal_var=False)
low, high = test.confidence_interval(confidence_level=0.95)

print("Difference of the mean slopes (winners - others):", round(winners.mean() - others.mean(), 4))
print("t statistic:", round(test.statistic, 2))
print("p-value:", round(test.pvalue, 3))
print("95% confidence interval of the difference:", round(low, 4), "to", round(high, 4))

Difference of the mean slopes (winners - others): -0.0175
t statistic: -2.0
p-value: 0.057
95% confidence interval of the difference: -0.0356 to 0.0005


## H. Comparison inside each ceremony

For each ceremony we compare the slope of the winner with the mean slope of the other nominees of the same ceremony. This gives one difference per ceremony (22 in total) and removes the differences between eras.

In [23]:
# Step 17: one difference per ceremony: slope of the winner minus mean slope of the other nominees
diffs = []
for ceremony, group in slopes.groupby("ceremony_year"):
    winner_slope = group.loc[group["is_winner"], "slope"].iloc[0]
    others_mean_slope = group.loc[~group["is_winner"], "slope"].mean()
    diffs.append({"ceremony_year": ceremony,
                  "nb_others": int((~group["is_winner"]).sum()),
                  "winner_slope": winner_slope,
                  "others_mean_slope": others_mean_slope,
                  "difference": winner_slope - others_mean_slope})

diffs = pd.DataFrame(diffs)
diffs.round(4)

,ceremony_year,nb_others,winner_slope,others_mean_slope,difference
0,1996,4,-0.0237,-0.0107,-0.0130
1,1997,4,-0.1170,-0.0352,-0.0818
2,1998,4,-0.0963,-0.0254,-0.0710
3,1999,4,-0.1007,-0.0291,-0.0716
4,2000,4,-0.0577,-0.0701,0.0123
5,2001,4,-0.0634,-0.0797,0.0163
6,2002,4,-0.0406,-0.0588,0.0182
7,2003,4,-0.0970,-0.0242,-0.0728
8,2004,4,-0.0381,-0.0392,0.0012
9,2005,4,-0.0462,-0.0467,0.0006


In [24]:
# Step 18: summary of the 22 differences (winner minus other nominees)
diffs["difference"].describe().round(4)

count    22.0000
mean     -0.0137
std       0.0334
min      -0.0818
25%      -0.0270
50%      -0.0021
75%       0.0104
max       0.0331
Name: difference, dtype: float64

In [25]:
# Step 19: is the mean difference different from 0? (one-sample t-test on the 22 differences, 95% confidence interval)
test = stats.ttest_1samp(diffs["difference"], 0)
low, high = test.confidence_interval(confidence_level=0.95)

print("Mean difference:", round(diffs["difference"].mean(), 4))
print("t statistic:", round(test.statistic, 2))
print("p-value:", round(test.pvalue, 3))
print("95% confidence interval:", round(low, 4), "to", round(high, 4))

Mean difference: -0.0137
t statistic: -1.93
p-value: 0.068
95% confidence interval: -0.0285 to 0.0011


In [26]:
# Step 20: sign count: in how many ceremonies is the winner's slope below / above the others', and the sign test
n_negative = int((diffs["difference"] < 0).sum())
n_positive = int((diffs["difference"] > 0).sum())
sign_test = stats.binomtest(n_negative, n_negative + n_positive, 0.5)

print("Winner declines faster (negative difference):", n_negative, "ceremonies")
print("Winner declines slower (positive difference):", n_positive, "ceremonies")
print("p-value of the sign test:", round(sign_test.pvalue, 3))

Winner declines faster (negative difference): 13 ceremonies
Winner declines slower (positive difference): 9 ceremonies
p-value of the sign test: 0.523


## Conclusion for question 1 (main analysis, 6 years from the ceremony)

Winners against the other nominees of the same ceremony, 22 ceremonies (1996 to 2017), weighted slope of the yearly average rating:
- mean difference: -0.0137 rating point per year (winners decline slightly faster); standard deviation of the 22 differences 0.0334, median -0.0021;
- t-test on the 22 differences: t = -1.93, p = 0.068, 95% confidence interval from -0.0285 to +0.0011;
- sign test: 13 ceremonies where the winner declines faster, 9 where it declines slower, p = 0.52.

We find no evidence that Best Picture winners keep their ratings better than the other nominees. The mean points slightly the other way but is compatible with chance, and it is carried by a few ceremonies with large negative differences (1997, 1998, 1999, 2003).

Limits: only 22 winners; the slope is a straight-line summary of six yearly averages; the audience of MovieLens changes over time and differs between films; no causal interpretation.

## I. Robustness of question 1

We repeat the comparison inside each ceremony with two other reasonable rules, to check that the conclusion does not depend on our choices:
- **A**: no weights (every year counts the same in the slope);
- **B**: rows with fewer than 100 ratings are removed (a film needs at least 3 years left to have a slope).

The main analysis (weights = number of ratings, all rows) is rerun with the same code to check that we get the same numbers.

In [27]:
# Step 21: the same chain as before, written once so that it can be repeated with different rules
import numpy as np

def film_slopes(rows_all, weighted=True):
    # one slope per film; a film needs at least 3 yearly rows
    results = []
    for movie_id, rows in rows_all.groupby("movieId"):
        if len(rows) < 3:
            continue
        X = sm.add_constant(rows["years_since_ceremony"])
        weights = rows["nb_ratings"] if weighted else np.ones(len(rows))
        fit = sm.WLS(rows["avg_rating"], X, weights=weights).fit()
        results.append({"movieId": movie_id, "slope": fit.params["years_since_ceremony"]})
    return films.merge(pd.DataFrame(results), on="movieId")

def winner_minus_others(slopes_table):
    # one difference per ceremony: slope of the winner minus mean slope of the other nominees
    differences = []
    for ceremony, group in slopes_table.groupby("ceremony_year"):
        winner = group.loc[group["is_winner"], "slope"]
        others = group.loc[~group["is_winner"], "slope"]
        if len(winner) == 1 and len(others) >= 1:
            differences.append(winner.iloc[0] - others.mean())
    return pd.Series(differences)

def summary(differences, label):
    test = stats.ttest_1samp(differences, 0)
    low, high = test.confidence_interval(confidence_level=0.95)
    n_neg = int((differences < 0).sum())
    n_pos = int((differences > 0).sum())
    sign_p = stats.binomtest(n_neg, n_neg + n_pos, 0.5).pvalue
    return {"rule": label, "ceremonies": len(differences), "mean_diff": round(differences.mean(), 4),
            "t": round(test.statistic, 2), "p_value": round(test.pvalue, 3),
            "ci_low": round(low, 4), "ci_high": round(high, 4),
            "negative": n_neg, "positive": n_pos, "sign_p": round(sign_p, 3)}

In [28]:
# Step 22: main analysis again, then variant A (no weights) and variant B (rows with fewer than 100 ratings removed)
rows_100 = rows_all[rows_all["nb_ratings"] >= 100]

table = pd.DataFrame([
    summary(winner_minus_others(film_slopes(rows_all, weighted=True)), "main: weights, all rows"),
    summary(winner_minus_others(film_slopes(rows_all, weighted=False)), "A: no weights"),
    summary(winner_minus_others(film_slopes(rows_100, weighted=True)), "B: at least 100 ratings"),
])
table

,rule,ceremonies,mean_diff,t,p_value,ci_low,ci_high,negative,positive,sign_p
0,"main: weights, all rows",22,-0.0137,-1.93,0.068,-0.0285,0.0011,13,9,0.523
1,A: no weights,22,-0.0143,-2.09,0.049,-0.0285,-0.0001,12,10,0.832
2,B: at least 100 ratings,22,-0.0136,-1.90,0.072,-0.0284,0.0013,13,9,0.523


## J. Question 2: the position of the winner among the nominees

Question 2: does the position of the winner among the nominees of its ceremony (ranked by average rating) improve, stay or get worse over the years after the ceremony?

For each ceremony we rank the nominees by average rating, in the year of the ceremony (year 0) and five years later (year 5).
The normalised rank is (number of nominees - rank) / (number of nominees - 1): 1 = best rated of the ceremony, 0 = worst, 0.5 = middle. Tied films get the average of their ranks.

In [29]:
# Step 23: rank and normalised rank of every nominee, in year 0 and in year 5, and the table of the 22 winners
ranks = con.sql("SELECT ceremony_year, title, is_winner, years_since_ceremony, avg_rating FROM film_year_window WHERE years_since_ceremony IN (0, 5)").df()

group = ranks.groupby(["ceremony_year", "years_since_ceremony"])
ranks["rank"] = group["avg_rating"].rank(ascending=False, method="average")
ranks["n_nominees"] = group["title"].transform("count")
ranks["normalised_rank"] = (ranks["n_nominees"] - ranks["rank"]) / (ranks["n_nominees"] - 1)

winners_year0 = ranks[ranks["is_winner"] & (ranks["years_since_ceremony"] == 0)].set_index("ceremony_year")
winners_year5 = ranks[ranks["is_winner"] & (ranks["years_since_ceremony"] == 5)].set_index("ceremony_year")

q2 = pd.DataFrame({"title": winners_year0["title"],
                   "n_nominees": winners_year0["n_nominees"],
                   "rank_year0": winners_year0["rank"],
                   "rank_year5": winners_year5["rank"],
                   "norm_rank_year0": winners_year0["normalised_rank"],
                   "norm_rank_year5": winners_year5["normalised_rank"]})
q2["change"] = q2["norm_rank_year5"] - q2["norm_rank_year0"]
q2.round(2)

,title,n_nominees,rank_year0,rank_year5,norm_rank_year0,norm_rank_year5,change
ceremony_year,,,,,,,
1996,Braveheart,5,1.0,1.0,1.00,1.00,0.00
1997,The English Patient,5,4.0,5.0,0.25,0.00,-0.25
1998,Titanic,5,3.0,5.0,0.50,0.00,-0.50
1999,Shakespeare in Love,5,2.0,4.0,0.75,0.25,-0.50
2000,American Beauty,5,2.0,1.0,0.75,1.00,0.25
2001,Gladiator,5,3.0,1.0,0.50,1.00,0.50
2002,A Beautiful Mind,5,2.0,2.0,0.75,0.75,0.00
2003,Chicago,5,3.0,5.0,0.50,0.00,-0.50
2004,The Lord of the Rings: The Return of the King,5,1.0,1.0,1.00,1.00,0.00
